# Datos masivos
## Clase 4. PySpark ML
###### Alberto Benavides

### Instalación de paquetes

In [ ]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
#Check this site for the latest download link https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!wget -q https://archive.apache.org/dist/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 https://cli.github.com/packages stable InRelease [3,917 B]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates/multiverse amd64 Packages [70.9 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,609 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,035 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd6

### Creación de la sesión

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
               .appName('ml') \
               .getOrCreate()

### Carga de datos

https://www.kaggle.com/datasets/meirnizri/covid19-dataset/data

In [ ]:
df = spark.read.csv(path='/content/Covid Data.csv', header = True)

### Exploración inicial

In [ ]:
df.printSchema()

root
 |-- USMER: string (nullable = true)
 |-- MEDICAL_UNIT: string (nullable = true)
 |-- SEX: string (nullable = true)
 |-- PATIENT_TYPE: string (nullable = true)
 |-- DATE_DIED: string (nullable = true)
 |-- INTUBED: string (nullable = true)
 |-- PNEUMONIA: string (nullable = true)
 |-- AGE: string (nullable = true)
 |-- PREGNANT: string (nullable = true)
 |-- DIABETES: string (nullable = true)
 |-- COPD: string (nullable = true)
 |-- ASTHMA: string (nullable = true)
 |-- INMSUPR: string (nullable = true)
 |-- HIPERTENSION: string (nullable = true)
 |-- OTHER_DISEASE: string (nullable = true)
 |-- CARDIOVASCULAR: string (nullable = true)
 |-- OBESITY: string (nullable = true)
 |-- RENAL_CHRONIC: string (nullable = true)
 |-- TOBACCO: string (nullable = true)
 |-- CLASIFFICATION_FINAL: string (nullable = true)
 |-- ICU: string (nullable = true)



- sex: 1 for female and 2 for male.
- age: of the patient.
- classification: covid test findings. Values 1-3 mean that the patient was diagnosed with covid in different
- degrees. 4 or higher means that the patient is not a carrier of covid or that the test is inconclusive.
- patient type: type of care the patient received in the unit. 1 for returned home and 2 for hospitalization.
- pneumonia: whether the patient already have air sacs inflammation or not.
- pregnancy: whether the patient is pregnant or not.
- diabetes: whether the patient has diabetes or not.
- copd: Indicates whether the patient has Chronic obstructive pulmonary disease or not.
- asthma: whether the patient has asthma or not.
- inmsupr: whether the patient is immunosuppressed or not.
- hypertension: whether the patient has hypertension or not.
- cardiovascular: whether the patient has heart or blood vessels related disease.
- renal chronic: whether the patient has chronic renal disease or not.
- other disease: whether the patient has other disease or not.
- obesity: whether the patient is obese or not.
- tobacco: whether the patient is a tobacco user.
- usmr: Indicates whether the patient treated medical units of the first, second or third level.
- medical unit: type of institution of the National Health System that provided the care.
- intubed: whether the patient was connected to the ventilator.
- icu: Indicates whether the patient had been admitted to an Intensive Care Unit.
- date died: If the patient died indicate the date of death, and 9999-99-99 otherwise.

In [ ]:
df.show()

+-----+------------+---+------------+----------+-------+---------+---+--------+--------+----+------+-------+------------+-------------+--------------+-------+-------------+-------+--------------------+---+
|USMER|MEDICAL_UNIT|SEX|PATIENT_TYPE| DATE_DIED|INTUBED|PNEUMONIA|AGE|PREGNANT|DIABETES|COPD|ASTHMA|INMSUPR|HIPERTENSION|OTHER_DISEASE|CARDIOVASCULAR|OBESITY|RENAL_CHRONIC|TOBACCO|CLASIFFICATION_FINAL|ICU|
+-----+------------+---+------------+----------+-------+---------+---+--------+--------+----+------+-------+------------+-------------+--------------+-------+-------------+-------+--------------------+---+
|    2|           1|  1|           1|03/05/2020|     97|        1| 65|       2|       2|   2|     2|      2|           1|            2|             2|      2|            2|      2|                   3| 97|
|    2|           1|  2|           1|03/06/2020|     97|        1| 72|      97|       2|   2|     2|      2|           1|            2|             2|      1|            1|    

### Preprocesamiento

In [ ]:
# Cambio de tipo de datos
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [ ]:
if False:
    columnas_a_entero = ["USMER", "MEDICAL_UNIT", "SEX", "PATIENT_TYPE", "INTUBED", "PNEUMONIA", "AGE",
                    "PREGNANT", "DIABETES", "COPD", "ASTHMA", "INMSUPR", "HIPERTENSION", "OTHER_DISEASE",
                    "CARDIOVASCULAR", "OBESITY", "RENAL_CHRONIC", "TOBACCO", "CLASIFFICATION_FINAL", "ICU"]

    for col in columnas_a_entero:
        df = df.withColumn(col, F.col(col).cast(T.IntegerType()))

    df = df.withColumn("DATE_DIED", F.when(F.col("DATE_DIED") != "9999-99-99", F.to_date(F.col("DATE_DIED"), "dd/MM/yyyy"))
                                    .otherwise(None))  # Reemplazar "9999-99-99" por nulos

In [ ]:
df.printSchema()

root
 |-- USMER: integer (nullable = true)
 |-- MEDICAL_UNIT: integer (nullable = true)
 |-- SEX: integer (nullable = true)
 |-- PATIENT_TYPE: integer (nullable = true)
 |-- DATE_DIED: date (nullable = true)
 |-- INTUBED: integer (nullable = true)
 |-- PNEUMONIA: integer (nullable = true)
 |-- AGE: integer (nullable = true)
 |-- PREGNANT: integer (nullable = true)
 |-- DIABETES: integer (nullable = true)
 |-- COPD: integer (nullable = true)
 |-- ASTHMA: integer (nullable = true)
 |-- INMSUPR: integer (nullable = true)
 |-- HIPERTENSION: integer (nullable = true)
 |-- OTHER_DISEASE: integer (nullable = true)
 |-- CARDIOVASCULAR: integer (nullable = true)
 |-- OBESITY: integer (nullable = true)
 |-- RENAL_CHRONIC: integer (nullable = true)
 |-- TOBACCO: integer (nullable = true)
 |-- CLASIFFICATION_FINAL: integer (nullable = true)
 |-- ICU: integer (nullable = true)



In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [ ]:
schema = StructType([
    StructField("USMER", IntegerType(), True),
    StructField("MEDICAL_UNIT", IntegerType(), True),
    StructField("SEX", IntegerType(), True),
    StructField("PATIENT_TYPE", IntegerType(), True),
    StructField("DATE_DIED", StringType(), True),
    StructField("INTUBED", IntegerType(), True),
    StructField("PNEUMONIA", IntegerType(), True),
    StructField("AGE", IntegerType(), True),
    StructField("PREGNANT", IntegerType(), True),
    StructField("DIABETES", IntegerType(), True),
    StructField("COPD", IntegerType(), True),
    StructField("ASTHMA", IntegerType(), True),
    StructField("INMSUPR", IntegerType(), True),
    StructField("HIPERTENSION", IntegerType(), True),
    StructField("OTHER_DISEASE", IntegerType(), True),
    StructField("CARDIOVASCULAR", IntegerType(), True),
    StructField("OBESITY", IntegerType(), True),
    StructField("RENAL_CHRONIC", IntegerType(), True),
    StructField("TOBACCO", IntegerType(), True),
    StructField("CLASIFFICATION_FINAL", IntegerType(), True),
    StructField("ICU", IntegerType(), True)
])

In [ ]:
df_schema = spark.read.csv(path='/content/Covid Data.csv', header = True, schema = schema)
df_schema.printSchema()

root
 |-- USMER: integer (nullable = true)
 |-- MEDICAL_UNIT: integer (nullable = true)
 |-- SEX: integer (nullable = true)
 |-- PATIENT_TYPE: integer (nullable = true)
 |-- DATE_DIED: string (nullable = true)
 |-- INTUBED: integer (nullable = true)
 |-- PNEUMONIA: integer (nullable = true)
 |-- AGE: integer (nullable = true)
 |-- PREGNANT: integer (nullable = true)
 |-- DIABETES: integer (nullable = true)
 |-- COPD: integer (nullable = true)
 |-- ASTHMA: integer (nullable = true)
 |-- INMSUPR: integer (nullable = true)
 |-- HIPERTENSION: integer (nullable = true)
 |-- OTHER_DISEASE: integer (nullable = true)
 |-- CARDIOVASCULAR: integer (nullable = true)
 |-- OBESITY: integer (nullable = true)
 |-- RENAL_CHRONIC: integer (nullable = true)
 |-- TOBACCO: integer (nullable = true)
 |-- CLASIFFICATION_FINAL: integer (nullable = true)
 |-- ICU: integer (nullable = true)



In [ ]:
# Mostrar el esquema actualizado
df_schema.show()

+-----+------------+---+------------+----------+-------+---------+---+--------+--------+----+------+-------+------------+-------------+--------------+-------+-------------+-------+--------------------+---+
|USMER|MEDICAL_UNIT|SEX|PATIENT_TYPE| DATE_DIED|INTUBED|PNEUMONIA|AGE|PREGNANT|DIABETES|COPD|ASTHMA|INMSUPR|HIPERTENSION|OTHER_DISEASE|CARDIOVASCULAR|OBESITY|RENAL_CHRONIC|TOBACCO|CLASIFFICATION_FINAL|ICU|
+-----+------------+---+------------+----------+-------+---------+---+--------+--------+----+------+-------+------------+-------------+--------------+-------+-------------+-------+--------------------+---+
|    2|           1|  1|           1|03/05/2020|     97|        1| 65|       2|       2|   2|     2|      2|           1|            2|             2|      2|            2|      2|                   3| 97|
|    2|           1|  2|           1|03/06/2020|     97|        1| 72|      97|       2|   2|     2|      2|           1|            2|             2|      1|            1|    

In [ ]:
# Crear una lista de filas que almacenen los valores distintos por cada columna
# excepto DATE_DIED
distinct_values = []

if True:
    for col in df_schema.columns:
        if col in ['DATE_DIED', 'AGE']:
            continue
        distinct_values.append((col, df_schema.select(col).distinct().rdd.flatMap(lambda x: x).collect()))
    print(distinct_values)

[('USMER', [1, 2]), ('MEDICAL_UNIT', [12, 1, 6, 3, 5, 9, 4, 8, 7, 10, 11, 2, 13]), ('SEX', [1, 2]), ('PATIENT_TYPE', [1, 2]), ('INTUBED', [1, 97, 2, 99]), ('PNEUMONIA', [1, 2, 99]), ('PREGNANT', [1, 97, 98, 2]), ('DIABETES', [1, 98, 2]), ('COPD', [1, 98, 2]), ('ASTHMA', [1, 98, 2]), ('INMSUPR', [1, 98, 2]), ('HIPERTENSION', [1, 98, 2]), ('OTHER_DISEASE', [1, 98, 2]), ('CARDIOVASCULAR', [1, 98, 2]), ('OBESITY', [1, 98, 2]), ('RENAL_CHRONIC', [1, 98, 2]), ('TOBACCO', [1, 98, 2]), ('CLASIFFICATION_FINAL', [1, 6, 3, 5, 4, 7, 2]), ('ICU', [1, 97, 2, 99])]


In [ ]:
columnas_con_nulos = ["SEX", "INTUBED", "PNEUMONIA",
                   "PREGNANT", "DIABETES", "COPD", "ASTHMA", "INMSUPR", "HIPERTENSION", "OTHER_DISEASE",
                   "CARDIOVASCULAR", "OBESITY", "RENAL_CHRONIC", "TOBACCO", "ICU"]
distinct_values = []

if True:
    for col in columnas_con_nulos:
        if col in ['DATE_DIED', 'AGE']:
            continue
        distinct_values.append((col, df_schema.select(col).distinct().rdd.flatMap(lambda x: x).collect()))
    distinct_values

In [ ]:
distinct_values

[('SEX', [1, 2]),
 ('INTUBED', [1, 97, 2, 99]),
 ('PNEUMONIA', [1, 2, 99]),
 ('PREGNANT', [1, 97, 98, 2]),
 ('DIABETES', [1, 98, 2]),
 ('COPD', [1, 98, 2]),
 ('ASTHMA', [1, 98, 2]),
 ('INMSUPR', [1, 98, 2]),
 ('HIPERTENSION', [1, 98, 2]),
 ('OTHER_DISEASE', [1, 98, 2]),
 ('CARDIOVASCULAR', [1, 98, 2]),
 ('OBESITY', [1, 98, 2]),
 ('RENAL_CHRONIC', [1, 98, 2]),
 ('TOBACCO', [1, 98, 2]),
 ('ICU', [1, 97, 2, 99])]

In [ ]:
# Reemplazar el valor "2" por "0" en cada columna de la lista
columnas_con_nulos = ["SEX", "INTUBED", "PNEUMONIA",
                   "PREGNANT", "DIABETES", "COPD", "ASTHMA", "INMSUPR", "HIPERTENSION", "OTHER_DISEASE",
                   "CARDIOVASCULAR", "OBESITY", "RENAL_CHRONIC", "TOBACCO", "ICU"]
for col in columnas_con_nulos:
    df_schema = df_schema.withColumn(col, F.when(F.col(col) == 2, 0).otherwise(F.col(col)))
    df_schema = df_schema.withColumn(col, F.when(F.col(col).isin(98, 99), None).otherwise(F.col(col)))

In [ ]:
columnas_con_nulos = ["SEX", "INTUBED", "PNEUMONIA",
                   "PREGNANT", "DIABETES", "COPD", "ASTHMA", "INMSUPR", "HIPERTENSION", "OTHER_DISEASE",
                   "CARDIOVASCULAR", "OBESITY", "RENAL_CHRONIC", "TOBACCO", "ICU"]
distinct_values = []

if True:
    for col in columnas_con_nulos:
        if col in ['DATE_DIED', 'AGE']:
            continue
        distinct_values.append((col, df_schema.select(col).distinct().rdd.flatMap(lambda x: x).collect()))
    distinct_values

https://www.kaggle.com/datasets/meirnizri/covid19-dataset/discussion/373795

- 97 en `PREGNANT` quiere decir que es un hombre
- 97 en `ICU` e `INTUBED` quiere decir que el paciente estaba en casa

Por simplicidad, se harán 0 los 97 de estas columnas.


In [ ]:
for col in ['PREGNANT', 'ICU', 'INTUBED']:
    df_schema = df_schema.withColumn(col, F.when(F.col(col) == 97, 0).otherwise(F.col(col)))

In [ ]:
# Cambio de nombre a columnas
df_schema = df_schema \
    .withColumnRenamed('SEX', 'WOMAN') \
    .withColumnRenamed('COPD', 'PULMONARY_CHRONIC')

In [ ]:
df_schema.printSchema()

root
 |-- USMER: integer (nullable = true)
 |-- MEDICAL_UNIT: integer (nullable = true)
 |-- WOMAN: integer (nullable = true)
 |-- PATIENT_TYPE: integer (nullable = true)
 |-- DATE_DIED: string (nullable = true)
 |-- INTUBED: integer (nullable = true)
 |-- PNEUMONIA: integer (nullable = true)
 |-- AGE: integer (nullable = true)
 |-- PREGNANT: integer (nullable = true)
 |-- DIABETES: integer (nullable = true)
 |-- PULMONARY_CHRONIC: integer (nullable = true)
 |-- ASTHMA: integer (nullable = true)
 |-- INMSUPR: integer (nullable = true)
 |-- HIPERTENSION: integer (nullable = true)
 |-- OTHER_DISEASE: integer (nullable = true)
 |-- CARDIOVASCULAR: integer (nullable = true)
 |-- OBESITY: integer (nullable = true)
 |-- RENAL_CHRONIC: integer (nullable = true)
 |-- TOBACCO: integer (nullable = true)
 |-- CLASIFFICATION_FINAL: integer (nullable = true)
 |-- ICU: integer (nullable = true)



También, por simplicidad, se crea una columna nueva `DIED` que indica si el paciente murió o no.

In [ ]:
df_schema = df_schema.withColumn(
    'DIED',
    F.when(F.col('DATE_DIED') != '9999-99-99', 1).otherwise(0)
)
# y se elimina 'DATE_DIED'
df_schema = df_schema.drop('DATE_DIED')

In [ ]:
df_schema.show()

+-----+------------+-----+------------+-------+---------+---+--------+--------+-----------------+------+-------+------------+-------------+--------------+-------+-------------+-------+--------------------+---+----+
|USMER|MEDICAL_UNIT|WOMAN|PATIENT_TYPE|INTUBED|PNEUMONIA|AGE|PREGNANT|DIABETES|PULMONARY_CHRONIC|ASTHMA|INMSUPR|HIPERTENSION|OTHER_DISEASE|CARDIOVASCULAR|OBESITY|RENAL_CHRONIC|TOBACCO|CLASIFFICATION_FINAL|ICU|DIED|
+-----+------------+-----+------------+-------+---------+---+--------+--------+-----------------+------+-------+------------+-------------+--------------+-------+-------------+-------+--------------------+---+----+
|    2|           1|    1|           1|      0|        1| 65|       0|       0|                0|     0|      0|           1|            0|             0|      0|            0|      0|                   3|  0|   1|
|    2|           1|    0|           1|      0|        1| 72|       0|       0|                0|     0|      0|           1|            0| 

### Preprocesamiento adicional

In [ ]:
from pyspark.ml.feature import StringIndexer, Imputer

En caso de que hubiera datos categóricos, se podría usar `StringIndexer` para asignarles un valor entero, susceptible a clasificación.

In [ ]:
datos_ejemplo = [
    ('Juan', 'Preparatoria', 15),
    ('María', 'Primaria', 8),
    ('Felipe', 'Secundaria', None),
    ('Nuria', 'Preparatoria', 15),
    ('Enrique', 'Universidad', 20),
    ('Juan', 'Preparatoria', None),
    ('Diana', 'Secundaria', 15),
]

df_ejemplo = spark.sparkContext.parallelize(datos_ejemplo).toDF(['nombre', 'escolaridad', 'edad'])
df_ejemplo.show()

+-------+------------+----+
| nombre| escolaridad|edad|
+-------+------------+----+
|   Juan|Preparatoria|  15|
|  María|    Primaria|   8|
| Felipe|  Secundaria|NULL|
|  Nuria|Preparatoria|  15|
|Enrique| Universidad|  20|
|   Juan|Preparatoria|NULL|
|  Diana|  Secundaria|  15|
+-------+------------+----+



In [ ]:
imputer = Imputer(strategy = 'mean', # median, mode
                  inputCols = ["edad"], outputCols = ["edad_imputada"])
df_imputado = imputer.fit(df_ejemplo).transform(df_ejemplo)

In [ ]:
df_imputado.show()

+-------+------------+----+-------------+
| nombre| escolaridad|edad|edad_imputada|
+-------+------------+----+-------------+
|   Juan|Preparatoria|  15|           15|
|  María|    Primaria|   8|            8|
| Felipe|  Secundaria|NULL|           14|
|  Nuria|Preparatoria|  15|           15|
|Enrique| Universidad|  20|           20|
|   Juan|Preparatoria|NULL|           14|
|  Diana|  Secundaria|  15|           15|
+-------+------------+----+-------------+



Claro, que aquí no tiene mucho sentido aplicarlo.

El `StringIndexer` indexa una columna categórica.

In [ ]:
indexer = StringIndexer(inputCol = "escolaridad", outputCol = "escolaridad_indexada")
df_indexado = indexer.fit(df_imputado).transform(df_imputado)

In [ ]:
df_indexado.printSchema()

root
 |-- nombre: string (nullable = true)
 |-- escolaridad: string (nullable = true)
 |-- edad: long (nullable = true)
 |-- edad_imputada: long (nullable = true)
 |-- escolaridad_indexada: double (nullable = false)



In [ ]:
df_indexado.show()

+-------+------------+----+-------------+--------------------+
| nombre| escolaridad|edad|edad_imputada|escolaridad_indexada|
+-------+------------+----+-------------+--------------------+
|   Juan|Preparatoria|  15|           15|                 0.0|
|  María|    Primaria|   8|            8|                 2.0|
| Felipe|  Secundaria|NULL|           14|                 1.0|
|  Nuria|Preparatoria|  15|           15|                 0.0|
|Enrique| Universidad|  20|           20|                 3.0|
|   Juan|Preparatoria|NULL|           14|                 0.0|
|  Diana|  Secundaria|  15|           15|                 1.0|
+-------+------------+----+-------------+--------------------+



In [ ]:
from pyspark.ml.feature import StandardScaler
from pyspark.ml.feature import VectorAssembler

In [ ]:
assembler = VectorAssembler().setInputCols(['edad_imputada', 'escolaridad_indexada']).setOutputCol("features")
df_features = assembler.transform(df_indexado)
df_features.show()

+-------+------------+----+-------------+--------------------+----------+
| nombre| escolaridad|edad|edad_imputada|escolaridad_indexada|  features|
+-------+------------+----+-------------+--------------------+----------+
|   Juan|Preparatoria|  15|           15|                 0.0|[15.0,0.0]|
|  María|    Primaria|   8|            8|                 2.0| [8.0,2.0]|
| Felipe|  Secundaria|NULL|           14|                 1.0|[14.0,1.0]|
|  Nuria|Preparatoria|  15|           15|                 0.0|[15.0,0.0]|
|Enrique| Universidad|  20|           20|                 3.0|[20.0,3.0]|
|   Juan|Preparatoria|NULL|           14|                 0.0|[14.0,0.0]|
|  Diana|  Secundaria|  15|           15|                 1.0|[15.0,1.0]|
+-------+------------+----+-------------+--------------------+----------+



In [ ]:
scaler = StandardScaler(inputCol="features", outputCol="features_scaled")
df_escalado = scaler.fit(df_features).transform(df_features)
df_escalado.show()

+-------+------------+----+-------------+--------------------+----------+--------------------+
| nombre| escolaridad|edad|edad_imputada|escolaridad_indexada|  features|     features_scaled|
+-------+------------+----+-------------+--------------------+----------+--------------------+
|   Juan|Preparatoria|  15|           15|                 0.0|[15.0,0.0]|[4.27948051618091...|
|  María|    Primaria|   8|            8|                 2.0| [8.0,2.0]|[2.28238960862982...|
| Felipe|  Secundaria|NULL|           14|                 1.0|[14.0,1.0]|[3.99418181510219...|
|  Nuria|Preparatoria|  15|           15|                 0.0|[15.0,0.0]|[4.27948051618091...|
|Enrique| Universidad|  20|           20|                 3.0|[20.0,3.0]|[5.70597402157455...|
|   Juan|Preparatoria|NULL|           14|                 0.0|[14.0,0.0]|[3.99418181510219...|
|  Diana|  Secundaria|  15|           15|                 1.0|[15.0,1.0]|[4.27948051618091...|
+-------+------------+----+-------------+---------

### Selección de características

In [ ]:
df_schema.printSchema()

root
 |-- USMER: integer (nullable = true)
 |-- MEDICAL_UNIT: integer (nullable = true)
 |-- WOMAN: integer (nullable = true)
 |-- PATIENT_TYPE: integer (nullable = true)
 |-- INTUBED: integer (nullable = true)
 |-- PNEUMONIA: integer (nullable = true)
 |-- AGE: integer (nullable = true)
 |-- PREGNANT: integer (nullable = true)
 |-- DIABETES: integer (nullable = true)
 |-- PULMONARY_CHRONIC: integer (nullable = true)
 |-- ASTHMA: integer (nullable = true)
 |-- INMSUPR: integer (nullable = true)
 |-- HIPERTENSION: integer (nullable = true)
 |-- OTHER_DISEASE: integer (nullable = true)
 |-- CARDIOVASCULAR: integer (nullable = true)
 |-- OBESITY: integer (nullable = true)
 |-- RENAL_CHRONIC: integer (nullable = true)
 |-- TOBACCO: integer (nullable = true)
 |-- CLASIFFICATION_FINAL: integer (nullable = true)
 |-- ICU: integer (nullable = true)
 |-- DIED: integer (nullable = false)



Se eliminan los valores nulos.

In [ ]:
df_dropna = df_schema.dropna()

In [ ]:
df_schema.count()

1048575

In [ ]:
df_dropna.count()

1019666

In [ ]:
df_dropna.count() / df_schema.count() * 100

97.24302028944042

In [ ]:
from pyspark.ml.feature import VectorAssembler, ChiSqSelector

In [ ]:
df_dropna.printSchema()

root
 |-- USMER: integer (nullable = true)
 |-- MEDICAL_UNIT: integer (nullable = true)
 |-- WOMAN: integer (nullable = true)
 |-- PATIENT_TYPE: integer (nullable = true)
 |-- INTUBED: integer (nullable = true)
 |-- PNEUMONIA: integer (nullable = true)
 |-- AGE: integer (nullable = true)
 |-- PREGNANT: integer (nullable = true)
 |-- DIABETES: integer (nullable = true)
 |-- PULMONARY_CHRONIC: integer (nullable = true)
 |-- ASTHMA: integer (nullable = true)
 |-- INMSUPR: integer (nullable = true)
 |-- HIPERTENSION: integer (nullable = true)
 |-- OTHER_DISEASE: integer (nullable = true)
 |-- CARDIOVASCULAR: integer (nullable = true)
 |-- OBESITY: integer (nullable = true)
 |-- RENAL_CHRONIC: integer (nullable = true)
 |-- TOBACCO: integer (nullable = true)
 |-- CLASIFFICATION_FINAL: integer (nullable = true)
 |-- ICU: integer (nullable = true)
 |-- DIED: integer (nullable = false)



Ahora se pueden seleccionar las características y la variable objetivo.

El `VectorAssambler` convierte las columnas dadas como entradas en una salida vectorial de características.

In [ ]:
inputCols = [
    'USMER',
    'MEDICAL_UNIT',
    'WOMAN',
    'PATIENT_TYPE',
    'INTUBED',
    #'PNEUMONIA',
    #'AGE',
    #'PREGNANT',
    #'DIABETES',
    #'PULMONARY_CHRONIC',
    #'ASTHMA',
    #'INMSUPR',
    #'HIPERTENSION',
    #'OTHER_DISEASE',
    #'CARDIOVASCULAR',
    #'OBESITY',
    #'RENAL_CHRONIC',
    #'TOBACCO',
    #'ICU',
    #'DIED'
]
assembler = VectorAssembler(inputCols = inputCols, outputCol = "features")
df_features = assembler.transform(df_dropna)

df_features.select('features').show(100, False)

+---------------------+
|features             |
+---------------------+
|[2.0,1.0,1.0,1.0,0.0]|
|[2.0,1.0,0.0,1.0,0.0]|
|[2.0,1.0,0.0,2.0,1.0]|
|[2.0,1.0,1.0,1.0,0.0]|
|[2.0,1.0,0.0,1.0,0.0]|
|[2.0,1.0,1.0,2.0,0.0]|
|[2.0,1.0,1.0,1.0,0.0]|
|[2.0,1.0,1.0,1.0,0.0]|
|[2.0,1.0,1.0,2.0,0.0]|
|[2.0,1.0,1.0,2.0,0.0]|
|[2.0,1.0,1.0,1.0,0.0]|
|[2.0,1.0,0.0,2.0,0.0]|
|[2.0,1.0,0.0,2.0,0.0]|
|[2.0,1.0,0.0,1.0,0.0]|
|[2.0,1.0,1.0,1.0,0.0]|
|[2.0,1.0,1.0,1.0,0.0]|
|[2.0,1.0,1.0,2.0,0.0]|
|[2.0,1.0,0.0,1.0,0.0]|
|[2.0,1.0,0.0,1.0,0.0]|
|[2.0,1.0,1.0,1.0,0.0]|
|[2.0,1.0,0.0,2.0,0.0]|
|[2.0,1.0,0.0,1.0,0.0]|
|[2.0,1.0,0.0,1.0,0.0]|
|[2.0,1.0,1.0,1.0,0.0]|
|[2.0,1.0,1.0,1.0,0.0]|
|[2.0,1.0,0.0,1.0,0.0]|
|[2.0,1.0,0.0,1.0,0.0]|
|[2.0,1.0,0.0,1.0,0.0]|
|[2.0,1.0,0.0,1.0,0.0]|
|[2.0,1.0,0.0,1.0,0.0]|
|[2.0,1.0,0.0,2.0,0.0]|
|[2.0,1.0,1.0,1.0,0.0]|
|[2.0,1.0,0.0,1.0,0.0]|
|[2.0,1.0,0.0,1.0,0.0]|
|[2.0,1.0,1.0,1.0,0.0]|
|[2.0,1.0,0.0,1.0,0.0]|
|[2.0,1.0,1.0,1.0,0.0]|
|[2.0,1.0,1.0,1.0,0.0]|
|[2.0,1.0,0.0,1.

In [ ]:
df_features.show(3)

+-----+------------+-----+------------+-------+---------+---+--------+--------+-----------------+------+-------+------------+-------------+--------------+-------+-------------+-------+--------------------+---+----+--------------------+
|USMER|MEDICAL_UNIT|WOMAN|PATIENT_TYPE|INTUBED|PNEUMONIA|AGE|PREGNANT|DIABETES|PULMONARY_CHRONIC|ASTHMA|INMSUPR|HIPERTENSION|OTHER_DISEASE|CARDIOVASCULAR|OBESITY|RENAL_CHRONIC|TOBACCO|CLASIFFICATION_FINAL|ICU|DIED|            features|
+-----+------------+-----+------------+-------+---------+---+--------+--------+-----------------+------+-------+------------+-------------+--------------+-------+-------------+-------+--------------------+---+----+--------------------+
|    2|           1|    1|           1|      0|        1| 65|       0|       0|                0|     0|      0|           1|            0|             0|      0|            0|      0|                   3|  0|   1|[2.0,1.0,1.0,1.0,...|
|    2|           1|    0|           1|      0|        1

Ahora se seleccionan las características según alguna métrica, en este caso $\tilde{\chi}^2$

In [ ]:
# Selección de características usando Chi-Square
selector = ChiSqSelector(numTopFeatures = 3, featuresCol = "features", labelCol="CLASIFFICATION_FINAL",
                         outputCol="selected_features")
df_sel = selector.fit(df_features).transform(df_features)
df_sel.show(5)

+-----+------------+-----+------------+-------+---------+---+--------+--------+-----------------+------+-------+------------+-------------+--------------+-------+-------------+-------+--------------------+---+----+--------------------+-----------------+
|USMER|MEDICAL_UNIT|WOMAN|PATIENT_TYPE|INTUBED|PNEUMONIA|AGE|PREGNANT|DIABETES|PULMONARY_CHRONIC|ASTHMA|INMSUPR|HIPERTENSION|OTHER_DISEASE|CARDIOVASCULAR|OBESITY|RENAL_CHRONIC|TOBACCO|CLASIFFICATION_FINAL|ICU|DIED|            features|selected_features|
+-----+------------+-----+------------+-------+---------+---+--------+--------+-----------------+------+-------+------------+-------------+--------------+-------+-------------+-------+--------------------+---+----+--------------------+-----------------+
|    2|           1|    1|           1|      0|        1| 65|       0|       0|                0|     0|      0|           1|            0|             0|      0|            0|      0|                   3|  0|   1|[2.0,1.0,1.0,1.0,...|   

### Estandarización

In [ ]:
from pyspark.ml.feature import StandardScaler

In [ ]:
scaler = StandardScaler(inputCol="features", outputCol="features_scaled")
df_escalado = scaler.fit(df_features).transform(df_features)

In [ ]:
df_escalado.show()

+-----+------------+-----+------------+-------+---------+---+--------+--------+-----------------+------+-------+------------+-------------+--------------+-------+-------------+-------+--------------------+---+----+--------------------+--------------------+
|USMER|MEDICAL_UNIT|WOMAN|PATIENT_TYPE|INTUBED|PNEUMONIA|AGE|PREGNANT|DIABETES|PULMONARY_CHRONIC|ASTHMA|INMSUPR|HIPERTENSION|OTHER_DISEASE|CARDIOVASCULAR|OBESITY|RENAL_CHRONIC|TOBACCO|CLASIFFICATION_FINAL|ICU|DIED|            features|     features_scaled|
+-----+------------+-----+------------+-------+---------+---+--------+--------+-----------------+------+-------+------------+-------------+--------------+-------+-------------+-------+--------------------+---+----+--------------------+--------------------+
|    2|           1|    1|           1|      0|        1| 65|       0|       0|                0|     0|      0|           1|            0|             0|      0|            0|      0|                   3|  0|   1|[2.0,1.0,1.0,1.

### Modelo

In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [ ]:
# Dividir datos en entrenamiento y prueba
train, test = df_escalado.randomSplit([0.7, 0.3])

# Crear y entrenar el modelo
lr = LogisticRegression(featuresCol="features_scaled", labelCol="CLASIFFICATION_FINAL")
model = lr.fit(train)

# Evaluar el modelo
predictions = model.transform(test)
evaluator = MulticlassClassificationEvaluator(labelCol="CLASIFFICATION_FINAL", predictionCol="prediction",
                                              metricName="accuracy")
accuracy = evaluator.evaluate(predictions)

In [ ]:
accuracy

0.5239628742358193


<details>
  <summary></summary>
    <img src='https://media1.tenor.com/m/DNCBqbguizsAAAAC/magic.gif'>
</details>

Otros ejemplos

In [ ]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import VectorAssembler

# Load and prepare the dataset
data = spark.read.csv("path/to/house_prices.csv", header=True, inferSchema=True)
feature_columns = ['size', 'bedrooms', 'age']  # assuming these are the features
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
data = assembler.transform(data)

# Split the data into training and test sets
train_data, test_data = data.randomSplit([0.7, 0.3])

# Train the model
lr = LinearRegression(featuresCol="features", labelCol="price")
model = lr.fit(train_data)

# Make predictions
predictions = model.transform(test_data)
predictions.select("prediction", "price").show()

In [ ]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import StringIndexer, VectorAssembler

# Load the Iris dataset
data = spark.read.csv("path/to/iris.csv", header=True, inferSchema=True)
data = StringIndexer(inputCol="species", outputCol="label").fit(data).transform(data)

# Assemble features
feature_columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
data = assembler.transform(data)

# Split the data
train_data, test_data = data.randomSplit([0.7, 0.3])

# Train the model
rf = RandomForestClassifier(featuresCol="features", labelCol="label")
model = rf.fit(train_data)

# Predictions
predictions = model.transform(test_data)
predictions.select("prediction", "label").show()

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler

# Assume data is already loaded and assembled as in the classification example
kmeans = KMeans().setK(3)
model = kmeans.fit(data)

# Make predictions
predictions = model.transform(data)
predictions.select("prediction").show()

## Tarea 5 y 6 (10 puntos c/u)
- Realizar análisis con MLlib de pyspark a tu conjunto de datos.
- Sube un reporte y tu código a tu repositorio.
- Presentar en la siguiente los hallazgos de tu tarea.
- Máximo 12 min de exposición por participante.